In [17]:
# Import Libraries
import ee
import geemap
import os
import json
from datetime import datetime
print("Libraries imported successfully")

Libraries imported successfully


In [18]:
CONFIG = {
    "region_name": "Mumbai",
    "roi": [72.77, 19.00, 73.00, 19.30],  # [min_lon, min_lat, max_lon, max_lat]
    "region" : ee.Geometry.Rectangle(CONFIG["roi"]),
    "dataset": "COPERNICUS/S2_SR_HARMONIZED",
    "past_start": "2023-06-01",
    "past_end": "2023-09-30",
    "recent_start": "2024-06-01",
    "recent_end": "2024-09-30",
    "cloud_threshold": 80,
    "scale": 30
}

In [19]:
# Initialize Earth Engine
ee.Initialize(project="gen-lang-client-0997797287")
print("Earth Engine Connected")

Earth Engine Connected


In [20]:
# Create folders if they don't exist
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../output", exist_ok=True)
print("Project directories ready")

Project directories ready


In [21]:
def fetch_image(start_date, end_date):
    collection = (
        ee.ImageCollection(CONFIG["dataset"])
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CONFIG["cloud_threshold"]))
    )
    
    size = collection.size().getInfo()
    
    if size == 0:
        raise ValueError(f"No images found between {start_date} and {end_date}")
    
    image = collection.median().clip(region)
    
    return image, size

In [23]:
# Fetch Past Image
past_image, past_count = fetch_image(
    CONFIG["past_start"],
    CONFIG["past_end"]
)

print("Past collection size:", past_count)

NameError: name 'region' is not defined

In [ ]:
# Fetch Recent Image
recent_image, recent_count = fetch_image(
    CONFIG["recent_start"],
    CONFIG["recent_end"]
)

print("Recent collection size:", recent_count)

In [ ]:
Map = geemap.Map(center=[19.1, 72.9], zoom=10)

Map.addLayer(
    past_image,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500},
    "Past Image"
)

Map.addLayer(
    recent_image,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500},
    "Recent Image"
)

Map.addLayer(region, {}, "ROI")

Map

In [ ]:
def export_image(image, filename):
    geemap.ee_export_image(
        image.select(["B3", "B8"]),  # Green + NIR
        filename=filename,
        scale=CONFIG["scale"],
        region=region,
        file_per_band=False
    )
    print(f"{filename} exported successfully")

In [ ]:
export_image(
    past_image,
    "../data/raw/past_image.tif"
)

export_image(
    recent_image,
    "../data/raw/recent_image.tif"
)

In [ ]:
metadata = {
    "region": CONFIG["region_name"],
    "dataset": CONFIG["dataset"],
    "past_period": [CONFIG["past_start"], CONFIG["past_end"]],
    "recent_period": [CONFIG["recent_start"], CONFIG["recent_end"]],
    "cloud_threshold": CONFIG["cloud_threshold"],
    "scale": CONFIG["scale"],
    "past_image_count": past_count,
    "recent_image_count": recent_count,
    "timestamp": datetime.utcnow().isoformat()
}

with open("../output/ingestion_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Ingestion metadata saved")